In [1]:
from functools import wraps
import time
import random
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [1]:
class CountCalls:
    def __init__(self, func):
        selaf.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call #{self.count}")
        return self.func(*args, **kwargs)

In [2]:
@CountCalls
def say_hello():
    print("Hello")

In [3]:
say_hello()

Call #1
Hello


In [4]:
say_hello()

Call #2
Hello


In [15]:
def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

In [16]:
@repeat(times=3)
def greet(name):
    print(f"Hello {name}")

greet("Harendra")

Hello Harendra
Hello Harendra
Hello Harendra


In [6]:
class HandleCorruption:
    def __init__(self, retries: int = 3, delay: int = 1):
        self.retries = retries
        self.delay = delay
    
    def __call__(self, func):
        def wrapper(*args, **kwargs):
            for attempt in range(1, self.retries + 1):
                try:
                    return func(*args, **kwargs)
                except (IOError, OSError, RuntimeError) as e:
                    print(f"Attempt {attempt} failed: {e}")
                    if attempt == self.retries:
                        raise
                    time.sleep(self.delay * attempt)
                except Exception as e:
                    print(f"Unexpected error: {e}")
                    raise
            return None  # This line is unreachable but kept for clarity
        return wrapper

In [7]:
@HandleCorruption(retries=3)
def read_csv_file() ->str:
    if random.random() < 0.8:
        raise RuntimeError("Corrupt file cannot be corrected after retries")
    else:
        print("File is not corrupted")

read_csv_file()

Attempt 1 failed: Corrupt file cannot be corrected after retries
Attempt 2 failed: Corrupt file cannot be corrected after retries
Attempt 3 failed: Corrupt file cannot be corrected after retries


RuntimeError: Corrupt file cannot be corrected after retries

In [8]:
def handle_corruption_file(retry: int, delay: int = 2):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, retry + 1):
                try:
                    return func(*args, **kwargs)
                except (IOError, OSError, RuntimeError) as e:  # Catch specific exceptions
                    print(f"Attempt {attempt} failed: {e}")
                    if attempt == retry:
                        raise
                    time.sleep(delay*attempt)
                except Exception as e:
                    # Don't retry on unexpected exceptions
                    print(f"Unexpected error: {e}")
                    raise
            raise RuntimeError("All retries failed")
        return wrapper
    return decorator

In [11]:
@handle_corruption_file(retry=3)
def read_file() ->str:
    if random.random() < 0.6:
        raise RuntimeError("Corrupt file detected")
    else:
        print("File is not corrupted")

In [12]:
read_file()

Attempt 1 failed: Corrupt file detected
Attempt 2 failed: Corrupt file detected
File is not corrupted
